# Regularization | Normalization | Learning Rate Scheduling
#### Abir Hossain | May, 2026
This notebook has the introductory definitions of methods in discussion. 
- [Regularization](#Regularization)
- [Normalization](#Normalization)
- [Learning Rate Scheduling](#Learning-Rate-Scheduling)

# Regularization
Regularization is any technique that constrains the model's capacity or biases the learning toward simpler solutions to improve generalization on unseen data. It combats the bias-variance tradeoff by preventing the model from memorizing the training set.  
Overfitting occurs when a model learns the ***noise*** in the training data rather than the underlying signal. This usually happens when the model is too complex *(too many parameters/weights)* relative to the amount of data. Regularization adds a penalty term to the loss function to constrain the model’s complexity. It forces the model to keep weights small or sparse, effectively simplifying the model.
- L2 Regularization (Ridge / Weight Decay)
- L1 Regularization (Lasso)
- Elastic Net
- Group Lasso / Structured Sparsity
- Orthogonal Regularization
- Spectral Normalization
- Dropout
- DropConnect
- SpatialDropout / DropBlock
- Stochastic Depth (DropPath)
- Cutout / GridMask / Random Erasing
- MixUp
- CutMix
- CutMix + MixUp
- AugMix / AutoAugment / RandAugment
- Label Smoothing
- Manifold Mixup
- FMix
- Early Stopping
- Gradient Clipping
- Gradient Penalty (WGAN-GP)
- Jacobian / Double Backprop Regularization
- VicReg (Variance-Invariance-Covariance)
- Stop Gradient
- Sharpness-Aware Minimization (SAM)
- ASAM (Adaptive SAM)
- ESAM / Efficient SAM
- Flooding
- Stochastic Weight Averaging (SWA)
- EMA (Exponential Moving Average) of Weights
- Lookahead
- Weight Standardization
- Sharpness-Aware Minimization with Momentum (MoSAM)
- Dropout for LLMs / Attention Dropout / Hidden Dropout
- Variational Dropout
- Concrete Dropout
- Bayesian Neural Networks (BBB, MC Dropout)
- Weight Uncertainty (Bayes by Backprop)

## L2 Regularization (Ridge Regression / Weight Decay)
L2 regularization penalizes the sum of the squared magnitudes of the weights. It encourages weights to be small and distributed across many features, but rarely exactly zero.
- **Mathematical Formulation:**  $R(\theta) = \|\theta\|_2^2 = \sum_{i=1}^{n} \theta_i^2$  
- **Full Loss Function:**  $J(\theta) = \text{MSE}(y, \hat{y}) + \lambda \sum_{i=1}^{n} \theta_i^2$  
- When the gradient of the L2 penalty with respect to a weight $\theta_i$ is taken:  
$\frac{\partial}{\partial \theta_i} (\lambda \theta_i^2) = 2\lambda \theta_i$   
During Gradient Descent, the update rule becomes:  
$\theta_i \leftarrow \theta_i - \eta \left( \frac{\partial \text{Loss}}{\partial \theta_i} + 2\lambda \theta_i \right)$  
$\theta_i \leftarrow \theta_i (1 - 2\eta\lambda) - \eta \frac{\partial \text{Loss}}{\partial \theta_i}$  
The term $(1 - 2\eta\lambda)$ is a factor less than 1. This means that at every step, the weight is multiplied by a number slightly smaller than 1, causing it to decay exponentially toward zero. This is why L2 is often called Weight Decay in deep learning frameworks *(PyTorch)*.
- **Geometric Interpretation:**
   + The L2 penalty creates a circular *(or spherical in higher dimensions)* constraint region around the origin.
   + The loss function contours are elliptical.
   + The optimal solution is where the loss contour first touches the circular constraint boundary.
   + Because the circle is smooth, it rarely touches the axes exactly. Thus, weights get very small but not exactly zero.
|Pros|Cons|
|---|---|
|Differentiable everywhere.|Does not produce sparse models *(all features preserved)*.|
|Stable numerical solution.|Can shrink important weights too much if $\lambda$ is high.|
|Handles multicollinearity well *(averages correlated features)*.|Not ideal for feature selection.|

# L1 Regularization (Lasso)
L1 regularization penalizes the sum of the absolute values of the weights. It encourages sparsity, meaning it drives many weights to exactly zero. This effectively performs feature selection.
- **Mathematical Formulation:**
$R(\theta) = \|\theta\|_1 = \sum_{i=1}^{n} |\theta_i|$
- **Full Loss Function:**
$J(\theta) = \text{MSE}(y, \hat{y}) + \lambda \sum_{i=1}^{n} |\theta_i|$  
The absolute value function $|x|$ is not differentiable at $x=0$. We use the subgradient:  
$\frac{\partial}{\partial \theta_i} (\lambda |\theta_i|) = \begin{cases} +\lambda & \text{if } \theta_i > 0 \\ -\lambda & \text{if } \theta_i < 0 \\ [-\lambda, +\lambda] & \text{if } \theta_i = 0 \end{cases}$   
During Gradient Descent:
$\theta_i \leftarrow \theta_i - \eta \left( \frac{\partial \text{Loss}}{\partial \theta_i} + \lambda \cdot \text{sign}(\theta_i) \right)$  
The penalty is a constant force ($\lambda$) pushing the weight toward zero, regardless of how small the weight already is.
    + If the weight is large, the data loss gradient might fight back.
    + If the weight is small, the constant $\lambda$ push dominates, forcing it to hit exactly zero and stay there.
- **Geometric Interpretation:**
    + The L1 penalty creates a diamond-shaped *(or polyhedral)* constraint region.
    + In 2D, the corners of the diamond lie on the axes.
    + The loss contours are likely to hit one of these corners first.
    + At a corner, one of the coordinates is zero. Hence, sparsity.
|Pros|Cons|
|---|---|
|Produces sparse models *(automatic feature selection)*.|Not differentiable at zero *(requires subgradient methods)*.|
|Interpretable models *(fewer active features)*.|Unstable if features are highly correlated *(arbitrarily picks one)*.|
|Robust to outliers.|Optimization can be slower/converge poorly if not handled carefully.|

## Elastic Net
Elastic Net is a hybrid that combines both L1 and L2 penalties. It aims to get the best of both worlds: the sparsity of L1 and the stability/correlation handling of L2. For two highly correlated features, L1 will arbitrarily pick one and set the other to zero. This is unstable *(small data changes might swap which one is picked)*. L2 shrinks correlated features together *(they get similar weights)*. It groups correlated features *(like L2)* but can still zero out entire groups if they are irrelevant *(like L1)*.
- **Mathematical Formulation:**
$R(\theta) = \alpha \sum_{i=1}^{n} |\theta_i| + (1-\alpha) \sum_{i=1}^{n} \theta_i^2$  
Or more commonly parameterized with two hyperparameters $\lambda_1$ and $\lambda_2$:  
$J(\theta) = \text{MSE}(y, \hat{y}) + \lambda_1 \|\theta\|_1 + \lambda_2 \|\theta\|_2^2$
    + $\lambda_1$: Controls L1 strength (sparsity).
    + $\lambda_2$: Controls L2 strength (shrinkage/stability).
- **Geometric Interpretation:**
    + The constraint region is a rounded diamond *(a square with rounded corners)*.
    + It has the corners of L1 *(allowing sparsity)* but the smoothness of L2 *(allowing stability)*.
|Pros|Cons|
|---|---|
|Handles correlated features better than L1.|Two hyperparameters to tune ($\lambda_1, \lambda_2$ or $\alpha, \lambda$).|
|Encourages sparsity.|More computationally expensive than pure L1 or L2.|
|Generally outperforms L1 or L2 alone in practice.|Slightly harder to interpret than pure L1.|

## Group Lasso / Structured Sparsity
Standard L1 regularization promotes element-wise sparsity *(individual weights become zero)*. However, in modern hardware *(GPUs/TPUs)*, removing individual weights doesn’t speed up inference because memory is accessed in blocks. Group Lasso promotes structured sparsity. It forces entire groups of parameters to become zero together.   
- **In CNNs:** This means removing entire filters/channels.
- **In RNNs/Transformers:** This can mean removing entire neurons or attention heads.  
This leads to actual model compression and faster inference, not just theoretical sparsity.
- **Mathematical Formulation:**
Instead of penalizing $\sum |w_i|$, we partition weights into disjoint groups $G_1, G_2, \dots, G_K$.  
$R(\theta) = \sum_{k=1}^{K} \sqrt{|G_k|} \cdot \|\theta_{G_k}\|_2$ 
    + $\theta_{G_k}$: The vector of weights in group kkk.
    + $||\theta_{G_k}\|_2$: The L2 norm of that group.
    + $\sqrt{|G_k|}$ : A normalization factor based on group size (to prevent bias toward larger groups).    
- We apply an L1 penalty on the L2 norms of groups.  
    + The outer L1 encourages some groups to be exactly zero.
    + The inner L2 encourages weights within a non-zero group to be small and distributed (like Ridge).
    + **Define Groups:** For a Conv layer with shape $(C_{out}, C_{in}, K, K)$, a group is usually one output filter $(C_{in}, K, K)$.
    + **Compute Group Norms:** Calculate the L2 norm for each filter.
    + **Apply Penalty:** Add the sum of these norms to the loss.
    + **Optimization:** During training, filters that contribute little to the loss will have their entire norm driven to zero.
    + **Pruning:** After training, you can physically remove the zeroed-out filters, resulting in a smaller, faster model.
- **Geometric Interpretation:**
    + Imagine a high-dimensional space where each axis represents a group *(not a single weight)*.
    + The penalty shape is a ***hyper-diamond*** in this group-space.
    + The solution hits the corners of this diamond, meaning entire groups are set to zero.
|Pros|Cons|
|---|---|
|***Hardware Friendly:*** Removes entire channels/filters, leading to real speedups.|***Hyperparameter Tuning:*** Choosing group structure and $\lambda$ is complex.|
|***Interpretability:*** Easier to see which features/channels are unused.|***Accuracy Drop:*** Aggressive pruning can hurt performance if not fine-tuned.|
|***Memory Efficiency:*** Reduces model size significantly.|***Training Instability:*** Can be harder to converge than standard L2.|

## Orthogonal Regularization
Neural networks often suffer from redundant features. If two neurons learn nearly identical features, they are wasting capacity. Orthogonal Regularization forces the weight vectors *(or filters)* to be orthogonal *(uncorrelated)* to each other. This encourages diversity in learned features, ensuring each neuron captures unique information.
- **Mathematical Formulation:**
Let $W$ be the weight matrix of a layer *(e.g., shape $D_{out} \times D_{in}$)*. We want the rows *(or columns)* of W to be orthogonal.  
Ideally, $W W^T = I$ *(Identity Matrix)*.  
The regularization penalty measures how far $W W^T$ is from I:  
$R(\theta) = \| W W^T - I \|_F^2$   
    + $\| \cdot \|_F$: Frobenius norm *(sum of squared elements)*.
    + I: Identity matrix.  
- **Alternative (Soft Orthogonality):**
Sometimes we only penalize the off-diagonal elements *(cosine similarity between different filters)*:
$R(\theta) = \sum_{i \neq j} \left( \frac{w_i \cdot w_j}{\|w_i\| \|w_j\|} \right)^2$
    + ***Forward Pass:*** Compute normal loss.
    + ***Regularization Term:*** Compute $W W^T$ and subtract Identity.
    + ***Backward Pass:*** Gradients flow back to push weight vectors apart if they are too similar.
    + ***Result:*** Filters become diverse. One might detect edges, another textures, another colors, rather than all detecting slightly shifted edges.
    + ***Prevents Collapse:*** In deep networks, layers can sometimes collapse to low-rank representations *(all neurons do the same thing)*. Orthogonality prevents this.
    + ***Better Conditioning:*** Orthogonal matrices preserve the norm of vectors ($\|Wx\| = \|x\|$). This helps prevent vanishing/exploding gradients in very deep networks.
    + ***Faster Convergence:*** Diverse features cover the input space more efficiently.
|Pros|Cons|
|---|---|
|***Feature Diversity:*** Prevents redundant neurons.|***Computational Cost:*** Computing $W W^T$ is $O(D^3)$ for large layers.|
|***Gradient Stability:*** Helps maintain signal magnitude in deep nets.|***Hard Constraint:*** Strict orthogonality can limit representational power.|
|***Interpretability:*** Features are less correlated.|***Not Always Beneficial:*** Some redundancy can be useful for robustness.|

## Spectral Normalization (SN)
Spectral Normalization controls the Lipschitz constant of a neural network.
- The Lipschitz constant $L$ measures how much the output can change relative to a small change in the input: $\|f(x) - f(y)\| \le L \|x - y\|$.
- If $L$ is too large, the function is sharp and unstable *(small input noise $\rightarrow$ huge output change)*.
- SN constrains the spectral norm *(largest singular value)* of each weight matrix to be 1. This ensures the network is ***1-Lipschitz (non-expansive)***.
- ***Primary Use Case:*** GANs. It stabilizes training by preventing the Discriminator from becoming too powerful/sharp.
- **Mathematical Formulation:**
For a weight matrix $W$, the spectral norm $\sigma(W)$ is its largest singular value.  
$\sigma(W) = \max_{h \neq 0} \frac{\|Wh\|_2}{\|h\|_2}$ 
- **Normalized Weight:**
$\hat{W} = \frac{W}{\sigma(W)}$

Computing SVD for every batch is too slow. SN uses ***Power Iteration*** to approximate the largest singular value efficiently.
- Initialize a random vector ***u*** *(and ***v***)*.  
- ***Iterate *(once per batch)*:***
  + $v \leftarrow \frac{W^T u}{\|W^T u\|_2}$
  + $u \leftarrow \frac{W v}{\|W v\|_2}$
- ***Estimate Spectral Norm:*** $\sigma(W) \approx u^T W v$
- ***Normalize:*** $\hat{W} = W / \sigma(W)$
- Use $\hat{W}$ in the forward pass.

This is a weight normalization technique, but unlike Batch Norm, it depends only on the weights, not the batch statistics. It is stable at inference time.
- ***Stabilizes GANs:*** Without SN, the Discriminator can become too confident *(gradients vanish)* or too chaotic *(gradients explode)*. SN keeps the Discriminator smooth.
- ***Prevents Exploding Gradients:*** By bounding the Lipschitz constant, gradients cannot grow exponentially through layers.
- ***No Hyperparameters:*** Unlike Batch Norm, SN has no running averages or momentum terms to tune.
|Pros|Cons|
|---|---|
|***GAN Stability:*** State-of-the-art for stabilizing GAN training.|***Capacity Reduction:*** Constraining weights can reduce model expressiveness.|
|***Computationally Cheap:*** Power iteration adds minimal overhead.|***Not Always Needed:*** For standard classification, L2/Dropout is often sufficient.|
|***Inference-Free:*** No batch-dependent statistics *(unlike BN)*.|***Approximation:*** Power iteration is an approximation *(though usually good enough)*.|

## Dropout
Dropout prevents co-adaptation of neurons. If neurons rely too heavily on specific neighbors, the model becomes fragile. By randomly dropping parts of the network during training, the network learns redundant, robust representations. It can be viewed as training an ensemble of $2^N$ sub-networks simultaneously.
- ***Mechanism:*** During training, each neuron *(activation)* is set to zero with probability ***p*** *(dropout rate)*. The remaining activations are scaled by $\frac{1}{1-p}$ *(inverted dropout)* to maintain expected value.
- **The Mathematical Formula:**
    + $\tilde{x}_i = \begin{cases} 0 & \text{with prob } p \\ \frac{x_i}{1-p} & \text{with prob } 1-p \end{cases}$ 
- ***Inference:*** No dropout is applied. The scaling is already handled during training *(inverted dropout)*.
- ***Best For:*** Fully Connected *(Dense)* layers.
- Standard default for most architectures.

## DropConnect
- ***Mechanism:*** Instead of dropping activations *(outputs of neurons)*, DropConnect drops weights *(connections between neurons)*. Each weight is set to zero with probability ***p***.
- ***Difference from Dropout:***
  + ***Dropout:*** $h = f(Wx) \cdot m$ *(mask applied to output)*.
  + ***DropConnect:*** $h = f((W \odot M)x)$ *(mask applied to weights)*.

- **Pros:** More fine-grained regularization than Dropout.
- **Cons:** Computationally expensive. You cannot use optimized matrix multiplication *(GEMM)* easily because the sparse mask changes every batch. Rarely used in practice today due to speed issues.
- Mostly of theoretical interest; standard Dropout is usually sufficient and faster.

## SpatialDropout / DropBlock (Vision-Specific)
- **Problem with Standard Dropout in CNNs:**
  + Convolutional layers have spatial correlation. Adjacent pixels share information.
  + If individual pixels/neurons are dropped randomly, the nearby pixels still contain almost the same information. The network can easily ignore the dropout because the signal is redundant locally.
- **SpatialDropout:** Drops entire feature maps *(channels)* or contiguous spatial regions.
- **DropBlock:** Drops a contiguous square block of neurons in the feature map.
- **Mechanism:** Select a center point, define a block size ($block\_size \times block\_size$), and zero out that region across all channels *(or specific channels)*.
- It forces the network to look at larger contexts. 
- Significantly better than standard Dropout for ResNets, EfficientNets, etc.

## Stochastic Depth (DropPath)
Used in Residual Networks (ResNets) and Transformers (ViT). Instead of dropping neurons, you drop entire layers/blocks.
- **The Mathematical Formula:**  
    $y = x + b_l \cdot F(x)$
    Where $b_l$ is a ***Bernoulli random variable*** *(0 or 1)* for layer l.
- During training, some residual blocks are skipped entirely *(identity mapping)*.
- This allows gradients to flow through shorter paths early in training *(easier optimization)*.
- As training progresses, deeper paths are gradually utilized.
- Standard in modern Vision Transformers *(e.g., DeiT, Swin Transformer)*.

## Cutout(Vision)
These techniques augment the input image directly by masking out regions. They teach the model to be robust to occlusion.
- **Mechanism:** Randomly select a square region in the input image and fill it with zeros *(or mean pixel value)*.
- Forces the model to not rely on a single distinctive feature *(e.g., just the dog's head)*. It must look at the whole body.
- Simple and effective.

## Random Erasing
- **Mechanism:** Similar to Cutout, but the aspect ratio and area of the erased rectangle are randomized.
- More diverse occlusions than fixed-square Cutout.
- Often used in Re-ID *(Person Re-Identification)* tasks.

## GridMask
- **Problem with Cutout/Erasing:** Erasing too much $\rightarrow$ lose all information. If you erase too little, it’s ineffective. It’s hard to tune.
- **Mechanism:** Applies a fixed grid pattern of masks. Some grid cells are kept, others are removed. The grid is rotated and shifted randomly.
- It preserves some information from every part of the image while removing enough to force robustness.
- State-of-the-art among simple masking augmentations.

## MixUp
These are advanced augmentations that generate new training samples by mixing two existing images and their labels. They regularize the model by encouraging linear behavior between classes. To train on convex combinations of pairs of examples.
- **The Mathematical Formula:**  
    $\tilde{x} = \lambda x_i + (1-\lambda) x_j$   
    $\tilde{y} = \lambda y_i + (1-\lambda) y_j$   
    $\lambda \sim \text{Beta}(\alpha, \alpha)$   $\; | \; (e.g., \alpha=0.2).$  
    $x_i, x_j$: Two random images.   
    $y_i, y_j$: Their one-hot labels.
- The resulting image looks like a ghostly overlay of two images.
- Encourages the model to behave linearly in between training examples.
- Reduces memorization of noisy labels.
- Improves calibration *(confidence matches accuracy)*.
- One of the most impactful regularizers for classification.

## CutMix
- **Problem with MixUp:** The mixed image is unnatural. The model might learn to recognize mixtures rather than real objects. Also, local features are preserved intact, so the model can still cheat by looking at small patches.This cuts a patch from one image and paste it onto another.
- **The Mathematical Formula:**  
  Generate a binary mask M *(0s and 1s)* representing a bounding box.  
  $\tilde{x} = M \odot x_i + (1-M) \odot x_j$    
  Labels are mixed proportionally to the area of the mask:   
  $\tilde{y} = \lambda y_i + (1-\lambda) y_j$   
  Where $\lambda$ is the ratio of pixels from $x_i$.   
- Looks like a collage. A dog’s head pasted onto a cat’s body.
- Forces the model to identify objects from partial views.
- More natural than MixUp (real pixels, not blurred overlays).
- Often outperforms MixUp in object detection and segmentation tasks.
- Standard in modern vision pipelines *(e.g., ResNet, ViT training)*.

## CutMix + MixUp (Combined)
Apply both augmentations in the same training pipeline.
- **Implementation:**
  + With probability $p_1$, apply MixUp.
  + With probability $p_2$, apply CutMix.
  + With probability $1-p_1-p_2$, use original image.
- MixUp encourages global linearity.
- CutMix encourages local robustness and partial object recognition.
- Together, they provide complementary regularization.
- Used in state-of-the-art models *(e.g., EfficientNet, RegNet, ViT)*.

## AutoAugment
Standard augmentations *(flip, crop, rotate)* require manual tuning of probabilities and magnitudes. These methods automate or optimize that process. Here, Reinforcement Learning is used to search for the optimal augmentation policy.
- **Mechanism:**
  + Define a search space of operations *(e.g., Shear, Translate, Rotate, Color)* and their possible magnitudes.
  + An RL agent *(controller)* proposes a policy *(a sequence of 2–5 operations with specific probabilities and magnitudes)*.
  + Train a small model using this policy.
  + The validation accuracy is the reward for the RL agent.
  + Repeat to find the best policy for the dataset *(e.g., ImageNet, CIFAR-10)*.
- **Pros:** State-of-the-art results when discovered.
- **Cons:**
  + **Expensive Search:** Requires thousands of GPU hours to find the policy.
  + **Dataset Specific:** A policy found for ImageNet may not work well for medical images or satellite data.

## RandAugment (RA)
Simplify AutoAugment by removing the RL search. Use a uniform random sampling strategy.
- **Mechanism:**
  + Define a fixed set of N augmentation operations.
  + For each image, randomly select n operations from the set.
  + Apply them with a global magnitude m *(tuned via simple grid search)*.
  + No probabilities to tune for individual operations.

The sheer diversity of random combinations acts as a strong regularizer.
- **Pros:**
  + **No Search Cost:** Zero RL training needed.
  + **Few Hyperparameters:** Only n *(number of ops)* and m *(magnitude)*.
  + **Generalizable:** Works well across different datasets without re-searching.
- The default choice for most modern vision tasks. Replaces AutoAugment in most pipelines.

## AugMix
Improve robustness and calibration *(uncertainty estimation)* by mixing multiple augmentation chains.
- **Mechanism:**
  + Generate K independent augmentation chains *(each chain is a sequence of 1–3 ops like RA/AA)*.
  + Apply each chain to the original image to get K augmented views.
  + Mix these K views with the original image using Dirichlet-distributed weights.
  + **Jensen-Shannon Divergence Loss:** Add a consistency loss that forces the model’s predictions on the mixed image to be close to the weighted average of predictions on the individual chains.
- Prevents the model from overfitting to specific augmentation artifacts.
- The JSD loss encourages smoothness in the prediction space.
- Best for safety-critical applications (medical, autonomous driving) where calibration matters.

## Label Smoothing
In standard classification, labels are one-hot encoded. This forces the model to predict probability 1.0 for the correct class and 0.0 for all others. This encourages overconfidence. The logits become arbitrarily large to minimize Cross-Entropy loss, leading to poor generalization and brittle predictions. The solution is to soften the targets.
- **The Mathematical Formulation:**
Let y be the one-hot vector, K be the number of classes, and $\epsilon$ be the smoothing factor *(e.g., 0.1)*.  
$y_{smoothed} = (1 - \epsilon) y + \frac{\epsilon}{K} \mathbf{1}$ 
    + **Loss Function:**
$L = - \sum_{k=1}^{K} y_{smoothed, k} \log(p_k)$

- The model is no longer rewarded for pushing the correct logit to infinity.
- It is penalized if it becomes too confident *(predicting 1.0)*.
- This keeps the logits bounded and the decision boundaries wider.
- Effectively acts as a regularizer on the output layer.
|Pros|Cons|
|---|---|
|Prevents overconfidence.|Slightly lower training accuracy but higher test acc.|
|Improves generalization.|Can hurt performance if $\epsilon$ is too large.|
|Simple to implement.|Not suitable for tasks requiring hard decisions.|

## Manifold Mixup
Standard MixUp interpolates inputs (x) and labels (y). Manifold Mixup interpolates hidden representations (h) inside the network.
- **Mechanism:**
    + Forward pass two samples $x_i, x_j$ through the network up to a random layer k.
    + Get hidden states $h_i^k, h_j^k$.
    + Interpolate hidden states:  
     $\tilde{h}^k = \lambda h_i^k + (1-\lambda) h_j^k$
    + Continue forward pass from layer $k+1$ using $\tilde{h}^k$.
    + Interpolate labels as usual: $\tilde{y} = \lambda y_i + (1-\lambda) y_j$.
    + Compute loss.
- The hidden space is non-linear. Interpolating in hidden space creates more diverse and complex synthetic samples than linear input interpolation.
- Forces intermediate representations to be smooth and linear between classes.
- Improves adversarial robustness more than input-level MixUp.

## FMix
MixUp and CutMix operate in pixel space. FMix operates in Fourier space *(frequency domain)*.
- **Mechanism:**
    + Compute 2D Discrete Fourier Transform of two images $x_i, x_j$.
    + Create a binary mask in the frequency domain *(low-pass or high-pass filter pattern)*.
    + Mix the Fourier coefficients:  
    $\tilde{X} = M \odot \text{DFT}(x_i) + (1-M) \odot \text{DFT}(x_j)$ 
    + Apply Inverse DFT to get the mixed image $\tilde{x}$.
    + Mix labels proportionally to the energy of the mask.
- Images have structure in the frequency domain *(edges are high-frequency, backgrounds are low-frequency)*.
- Mixing in Fourier space preserves global structure while swapping textures/details.
- Complements CutMix *(which swaps spatial patches)* by swapping frequency components.

## Early Stopping
Stop training when the model starts to overfit.
- Monitor Validation Loss *(not Training Loss)*.
- If Val Loss stops decreasing *(or starts increasing)* for N consecutive epochs *(patience)*, halt training.
- Restore weights from the epoch with the best Val Loss.
- Training loss usually decreases monotonically.
- Validation loss decreases initially, then plateaus, then increases *(overfitting)*.
- Early stopping finds the sweet spot between underfitting and overfitting.
- Acts as an implicit regularizer *(limits the number of optimization steps)*.
|Pros|Cons|
|---|---|
|Prevents overfitting automatically.|Requires a validation set.|
|Saves computation time.|May stop too early if Val Loss is noisy.|
|Simple to implement.|Not compatible with some LR schedules unless adjusted.|

## Gradient Clipping
Prevent exploding gradients by capping the norm of the gradient vector.
- **Two common methods:**
    + **Clip by Value:** Clamp each gradient element to [−c,c].
    + **Clip by Norm:** Scale the entire gradient vector if its norm exceeds a threshold c.  
    $g \leftarrow g \cdot \min\left(1, \frac{c}{\|g\|}\right)$
- In RNNs and Transformers, gradients can accumulate exponentially through time/layers.
- Large gradients cause huge weight updates, leading to NaNs or divergence.
- Clipping ensures updates remain in a safe region

## Gradient Penalty (WGAN-GP)
Used in Wasserstein GANs *(WGAN)* to enforce the Lipschitz constraint on the Discriminator. Original WGAN used Weight Clipping *(clamp weights to [−0.01,0.01])*. This is crude and leads to capacity issues. WGAN-GP replaces clipping with a gradient penalty.
- **The Mathematical Formulation:**  
    + The Discriminator D must satisfy $\|\nabla_x D(x)\|_2 \le 1$ everywhere.
    + We penalize deviations from this constraint on interpolated points $\hat{x}$ between real and fake data.  
$L_{GP} = \lambda \mathbb{E}_{\hat{x}} \left[ (\|\nabla_{\hat{x}} D(\hat{x})\|_2 - 1)^2 \right]$  
   $\hat{x} = \epsilon x_{real} + (1-\epsilon) x_{fake}$  
    $\lambda$: Penalty coefficient *(usually 10)*.
- Sample real and fake images.
- Interpolate them to create $\hat{x}$.
- Compute gradients of $D(\hat{x})$ w.r.t $\hat{x}$.
- Add penalty if gradient norm $\neq 1$.
|Pros|Cons|
|---|---|
|Stable GAN training.|Computationally expensive *(requires 2nd backward pass)*.|
|Better sample quality than Weight Clipping.|Slower than Spectral Normalization.|
|Theoretically sound.|

## Jacobian & Double Backprop Regularization
Standard regularization penalizes the weights ($W$). Jacobian regularization penalizes the sensitivity of the output to the input. It forces the model to be smooth and robust to small input perturbations *(noise)*.
- **The Mathematical Formulation:**
Let $J_f(x)$ be the Jacobian matrix of the network output $f(x)$ with respect to input x.  
$J_f(x) = \frac{\partial f(x)}{\partial x}$  
- **Jacobian Regularization Loss:**  
$L_{jac} = \| J_f(x) \|_F^2 = \sum_{i,j} \left( \frac{\partial f_i}{\partial x_j} \right)^2$   
- **Double Backprop:** Computing the full Jacobian is expensive. Double Backprop usually refers to computing the gradient of the loss with respect to the input, then regularizing that gradient.  
$L_{db} = \| \nabla_x L(f(x), y) \|_2^2$
- **Forward Pass:** Compute loss LLL.
- **First Backward:** Compute $\nabla_x L$ *(gradient of loss w.r.t input)*.
- **Regularization:** Add $\|\nabla_x L\|^2$ to the total loss.
- **Second Backward:** Compute gradients of this new term w.r.t weights W. This requires differentiating through the gradient operation *(Hessian-vector product)*.
|Pros|Cons|
|---|---|
|Makes the model less sensitive to adversarial attacks.|Very expensive *(requires 2nd-order derivatives)*.|
|Prevents sharp, jagged decision boundaries.|Stores intermediate gradients for backprop-through-backprop.|
|Encourages simplicity in the input-output mapping.|Not supported by all optimizers/frameworks easily.|

## VicReg (Variance-Invariance-Covariance)
VicReg is a Self-Supervised Learning objective that prevents model collapse *(where all outputs become identical)* without using negative samples or asymmetric architectures. It explicitly enforces three properties on the batch of embeddings.  
- **Mechanism:**
Let Z be the batch of embeddings (size $N \times D$).  
    + **Invariance (MSE):**  
        Two augmented views of the same image should have similar embeddings.  
        $L_{inv} = \text{MSE}(z_i, z'_i)$   
    + **Variance (Std Dev):**  
        Each feature dimension across the batch should have unit variance. Prevents all outputs from collapsing to a constant.
        $L_{var} = \sum_{j=1}^{D} \max(0, 1 - \sqrt{\text{Var}(z_{:,j}) + \epsilon})$   
    + **Covariance (Decorrelation):**  
        Different feature dimensions should be uncorrelated. Prevents redundant features.
        $L_{cov} = \sum_{i \neq j} C_{ij}^2 \quad \text{where } C \text{ is the covariance matrix of } Z$
    + **Total Loss:**  
$L = \lambda L_{inv} + \mu L_{var} + \nu L_{cov}$
- Unlike Contrastive Learning, it doesn't require large batches of negative samples.
- Directly optimizes the statistical properties of the representation space.
- Less prone to collapse than simpler SSL methods.

## Stop Gradient
Prevent gradients from flowing through specific parts of the computational graph. This is not a regularizer in the traditional sense, but a structural constraint used in SSL and Multi-Task Learning. The forward pass uses the value of z, but the backward pass treats z as a constant (gradient = 0).

## Sharpness-Aware Minimization (SAM)
Standard SGD finds minima with low loss. SAM finds minima with low loss AND low curvature *(flat minima)*.
- **Flat Minima:** Generalize better because small shifts in data *(test set)* don’t change the loss much.
- **Sharp Minima:** Overfit; small data shifts cause large loss spikes.
- **The Mathematical Formulation:**  
SAM minimizes the worst-case loss within a neighborhood of radius $\rho$.  
$\min_w \max_{\|\epsilon\|_2 \le \rho} L(w + \epsilon)$  
- **Two-Step Update:**
    + **Lookahead:** Find the perturbation $\epsilon$ that maximizes loss:  
    $\epsilon = \rho \frac{\nabla_w L(w)}{\|\nabla_w L(w)\|_2}$   
    + **Update:** Compute gradient at the perturbed location $w + \epsilon$ and update original weights w:  
    $w \leftarrow w - \eta \nabla_w L(w + \epsilon)$ 
- By looking at the peak of the loss landscape nearby, SAM avoids sharp peaks.
- It effectively smooths the loss landscape during training.
|Pros|Cons|
|---|---|
|***SOTA Generalization:*** Consistently beats SGD/Adam on ImageNet, CIFAR.|***2x Computational Cost:*** Requires two forward/backward passes per step.|
|***Robustness:*** Better performance on shifted/corrupted data.|***Hyperparameter Sensitivity:*** $\rho$ needs tuning.|
|***Label Noise Robustness:*** Less likely to memorize noisy labels.|

## ASAM (Adaptive SAM) & ESAM
- **ASAM (Adaptive SAM):**
    + **Problem with SAM:** The perturbation radius $\rho$ is fixed for all weights. But weights have different scales *(e.g., biases vs. conv kernels)*.
    + **Solution:** Scale the perturbation adaptively based on the weight magnitude.  
    $\epsilon_i = \rho \frac{|\nabla_i L|}{\sqrt{\sum (\nabla L)^2}} \cdot \frac{1}{|w_i| + \delta}$  
    + **Benefit:** More stable and often outperforms standard SAM.

- **ESAM (Efficient SAM):**
    + **Problem with SAM:** 2x cost is prohibitive for large models.
    + **Solution:** Approximate SAM. Use sharpness-aware updates only for a subset of layers or steps. Or, use a cheaper approximation of the gradient norm.
    + **Benefit:** Near-SAM performance with ~1.2x cost.

## Flooding
Standard training drives loss to zero. Flooding intentionally keeps the loss above a small threshold b. If $L > b$: Minimize loss normally. If $L < b$: Maximize loss *(push it back up to b)*.
- **Mechanism:**
$L_{flood} = |(L(w) - b)| + b$ 
*(Note: Implementation often uses `abs(L - b) + b` or `sign-based switching`)*   
- Prevents the model from memorizing individual training examples *(which drives loss to near-zero)*.
- Forces the model to wander around the flat basin of loss b, finding smoother solutions.
- Acts as a regularizer against overfitting to noise.

## Stochastic Weight Averaging (SWA)
Instead of taking the final weights after training, average the weights visited during the last few epochs. SGD trajectories often wander around a wide, flat minimum. The center of this trajectory is often a better generalizer than any single point.
- **Mechanism:**
    + Train model with standard SGD/Cyclical LR.
    + After epoch $N_{start}$, start collecting weights $w_i$ every K epochs.
    + Maintain a running average:  
    $w_{SWA} = \frac{1}{M} \sum_{i=1}^{M} w_i$   
    + Use $w_{SWA}$ for evaluation/inference.
|Pros|Cons|
|---|---|
|***Free Performance Boost:*** Often 0.5–1% accuracy gain.|***Batch Norm Issues:*** SWA weights change the distribution; BN stats must be recalculated.|
|***Robust:*** Less sensitive to final epoch luck.|***Inference Only:*** Cannot continue training from SWA weights easily.|
|***Simple:*** Easy to implement.|

## EMA (Exponential Moving Average) of Weights
Maintain a shadow copy of the weights that updates slowly using an exponential moving average.  
$w_{EMA} \leftarrow \alpha w_{EMA} + (1-\alpha) w_{current}$   
$\alpha \approx 0.999\;$ (very slow update).  
- ***Smoothing:*** Filters out high-frequency noise in weight updates.
- ***Stability:*** The EMA model is often more stable and generalizes better than the instantaneous model.
- ***Standard in:***
  + Generator EMA is critical for high-quality image synthesis (StyleGAN, BigGAN).
  + EMA weights are used for sampling.
  + Target networks in BYOL/DINO are essentially EMA models.
- ***SWA:*** Uniform average of discrete checkpoints. Best for classification.
- ***EMA:*** Continuous exponential average. Best for generative models and SSL.

## Lookahead Optimizer
Lookahead is a meta-optimizer that wraps around any base optimizer. It maintains two sets of weights:
- ***Fast Weights:*** Updated by the base optimizer for k steps.
- ***Slow Weights:*** Updated once every k steps by moving towards the final fast weights.
It acts as a trajectory regularizer, smoothing out the optimization path and reducing variance in weight updates.
- **Mechanism:** Initialize slow weights $w_{slow}$ and fast weights $w_{fast} = w_{slow}$.  
    + **Inner Loop (k steps):**  
        Update $w_{fast}$ using base optimizer on mini-batches.   
    + **Outer Loop (1 step):**   
        * ***Update slow weights:***   
        $w_{slow} \leftarrow w_{slow} + \alpha (w_{fast} - w_{slow})$
        * ***Reset fast weights:*** $w_{fast} \leftarrow w_{slow}$.
        * ***$\alpha$:*** Lookahead step size *(usually 0.5)*.
        * ***k:*** Sync period *(usually 5–20)*.
- The fast weights explore the loss landscape aggressively.
- The slow weights act as a stable anchor, preventing the model from diverging or getting stuck in sharp minima.
- Effectively averages the trajectory, similar to SWA but during training.
|Pros|Cons|
|---|---|
|Reduces oscillation in loss.|Adds $\alpha$ and k to tune.|
|Often finds flatter minima.|Stores two copies of weights.|
|Works with any optimizer.|Less impactful with modern adaptive optimizers.|

## Weight Standardization (WS)
Batch Normalization has issues with small batch sizes and RNNs. Weight Standardization replaces BN by normalizing the weights themselves before convolution. It ensures that the output of each layer has zero mean and unit variance, independent of batch statistics.
- **The Mathematical Formulation:**
For a convolutional filter W, compute mean and variance across spatial and input channel dimensions:  
    $\mu_W = \frac{1}{C_{in} K^2} \sum W$  
    $\sigma_W^2 = \frac{1}{C_{in} K^2} \sum (W - \mu_W)^2$   
Standardize weights:  
    $\hat{W} = \frac{W - \mu_W}{\sqrt{\sigma_W^2 + \epsilon}}$  
    Use $\hat{W}$ for convolution.  
- ***Decouples Scale and Direction:*** Optimization becomes easier because the gradient direction is normalized.
- ***Batch-Size Independent:*** Unlike BN, WS works perfectly with batch size = 1.
- ***Used in NFNet:*** Normalizer-Free Networks use WS + Adaptive Gradient Clipping to achieve SOTA without any normalization layers.
|Pros|Cons|
|---|---|
|Works with small batches/RNNs.|Computes stats per forward pass.|
|Simplifies architecture *(no running stats)*.|Requires careful initialization.|
|Helps train very deep networks.|

## Sharpness-Aware Minimization with Momentum (MoSAM)
Standard SAM uses the current gradient to find the perturbation. MoSAM incorporates momentum into the perturbation calculation. SAM can be noisy because the gradient direction changes rapidly between batches. MoSAM smooths the gradient estimate used for the lookahead step.
- **Mechanism:**
    + Maintain a momentum buffer $m_t$ for gradients.  
    + Compute perturbation $\epsilon$ using the momentum-adjusted gradient instead of the raw instantaneous gradient.  
    $\epsilon \propto \text{Normalize}(m_t)$  
    + Compute loss at $w + \epsilon$ and update weights.
- More stable perturbation direction.
- Less sensitive to batch noise.
- Often converges faster than vanilla SAM.

## Dropout for LLMs / Transformers
Standard Dropout is applied in three specific places in Transformer architectures.
- **Attention Dropout:** Applied to the attention weights *(softmax output)* before multiplying with values. Randomly ignores some token-to-token connections. Forces the model to not rely on a single attention head or key-value pair.
- **Hidden Dropout (Residual Dropout):** Applied to the output of the Feed-Forward Network *(FFN)* or the Attention Output before adding to the residual stream. Regularizes the transformation within each block.
- **Embedding Dropout:** Applied to the input token embeddings. Forces the model to handle missing or noisy input tokens robustly.
- **Lower Rates:** LLMs often use lower dropout rates *(0.0–0.1)* compared to CNNs *(0.5)* because they are trained on massive datasets where overfitting is less of a concern than underfitting.
- **DropPath:** In Vision Transformers *(ViT)*, Stochastic Depth *(DropPath)* is preferred over standard dropout for entire blocks.

## Variational Dropout
Standard Dropout uses a fixed dropout rate ppp. Variational Dropout treats the dropout rate as a learnable parameter for each neuron or weight, optimized via Variational Inference. It allows the model to decide which neurons are important *(low dropout)* and which are redundant *(high dropout)*.
- **Mechanism:**
    + Assign a variational parameter $\theta_i$ *(log-alpha)* to each weight/neuron.
    + The dropout rate $p_i$ is derived from $\theta_i$ via a sigmoid or softplus function.
    + Optimize $\theta_i$ alongside weights W using ***Evidence Lower Bound (ELBO)*** loss.  
    $L = \mathbb{E}_{q}[\log P(D|W)] - KL(q(W|\theta) || P(W))$
- Neurons with high learned dropout rates can be pruned after training.
- Stronger regularization where needed, weaker where data is abundant.
- For automatic model compression.

## Concrete Dropout
Variational Dropout is hard to optimize because dropout is discrete *(0 or 1)*. Concrete Dropout uses the Concrete Distribution *(a continuous relaxation of the Bernoulli distribution)* to make dropout rates differentiable.
- **Mechanism:**
    + Sample $u \sim \text{Uniform}(0, 1)$.
    + Compute concrete variable:  
    $z = \sigma\left(\frac{\log u - \log(1-u) + \log p}{\tau}\right)$   
       * p: Learnable dropout probability.
       * $\tau$: Temperature parameter *(annealed during training)*.
    As $\tau \to 0$, z becomes binary *(0 or 1)*.
    + Gradients flow through z to update p.
- Allows end-to-end learning of dropout rates via backprop.
- No need for separate variational inference steps.
- Automatically tunes regularization strength per layer.

## Bayesian Neural Networks (BNNs) & MC Dropout
Standard NNs give point estimates. BNNs place a probability distribution over weights P(W∣D)P(W|D)P(W∣D). This allows the model to output uncertainty (confidence intervals).
- **Bayes by Backprop:**
    + Approximate the posterior $P(W∣D)$ with a variational distribution $q(W|\theta)$ *(e.g., Gaussian)*.
    + Minimize KL divergence between $q(W)$ and prior $P(W)$, while maximizing likelihood.
    + Requires sampling weights for every forward pass.
- **Monte Carlo (MC) Dropout:**
    + Dropout at Inference Time.
    + **Mechanism:**
        * Train a standard network with Dropout.
        * At test time, keep Dropout active.
        * Run T forward passes with different dropout masks.
        * Compute mean and variance of predictions.  
        $\mu = \frac{1}{T} \sum \hat{y}_t, \quad \sigma^2 = \frac{1}{T} \sum (\hat{y}_t - \mu)^2$ 
- *Gal & Ghahramani (2016)* proved that MC Dropout is a variational approximation to a Gaussian Process. It provides free uncertainty estimation without changing the architecture.
|Pros|Cons|
|---|---|
|Distinguishes aleatoric *(data)* vs. epistemic *(model)* uncertainty.|Requires T forward passes (10–50x slower).|
|Just keep dropout on at test time.|Not a true Bayesian posterior.|
|Better *Out-of-Distribution* detection.|

## Weight Uncertainty (Bayes by Backprop)
Explicitly learn a distribution for each weight $w_i \sim \mathcal{N}(\mu_i, \sigma_i^2)$. Instead of learning a single value w, learn $\mu$ *(mean)* and $\rho$ *(variance parameter)*.
- Sample weights during training:
  $w = \mu + \sigma \odot \epsilon$
- **Computational Cost:** Doubles the number of parameters ($\mu$ and $\sigma$ for every weight).
- **Sampling Noise:** Requires multiple samples per batch to estimate gradients accurately.
- **Convergence Issues:** Harder to optimize than standard SGD.

# Normalization
Normalization stabilizes the distribution of layer inputs during training. It reduces internal covariate shift (or more accurately, smooths the optimization landscape) by ensuring activations have controlled mean and variance.   
Internal Covariate Shift is the historical justification, but practically, normalization smooths the optimization landscape. It allows for higher learning rates and reduces sensitivity to initialization by ensuring activations stay within a manageable range *(usually zero-mean, unit-variance)* before passing through non-linearities. All these methods generally perform two steps:
- **Normalization:** Subtract mean, divide by standard deviation.
- **Affine Transformation:** Learnable scale ($\gamma$) and shift ($\beta$) parameters to restore representational power if necessary.  
$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$  
$y = \gamma \hat{x} + \beta$  
The difference between the methods below is which dimensions are used to calculate $\mu$ and $\sigma$.   
- Batch Normalization
- Layer Normalization 
- Instance Normalization 
- Group Normalization 
- RMSNorm (Root Mean Square Norm)
- Pre-Norm vs Post-Norm
- DeepNorm
- Sandwich Norm
- ScaleNorm
- Batch Renormalization
- Switchable Norm
- Filter Response Normalization 
- Online Normalization
- Cross-Layer Normalization
- PowerNorm
- Normalizer-Free Networks (NFNet)
- ReZero / SkipInit
- Fixup Initialization
- ActNorm (Glow)

## Batch Normalization (BatchNorm)
Normalizes across the batch dimension for each feature channel independently.
- **Input Shape:** (N,C,H,W) for ConvNets or (N,T,D) for sequences.
- **Statistics Calculation:**  
        $\mu_c = \frac{1}{N \cdot H \cdot W} \sum_{n,h,w} x_{n,c,h,w}$  
        $\sigma^2_c = \frac{1}{N \cdot H \cdot W} \sum_{n,h,w} (x_{n,c,h,w} - \mu_c)^2$  
- **Process:**
  + Compute mean/variance per channel C across the entire mini-batch N and spatial dimensions H,WH, WH,W.
  + Normalize.
  + Apply learnable $\gamma_c, \beta_c$ per channel.
  + Inference Mode: Uses running averages of mean/variance tracked during training *(exponential moving average)*.
- **Pros:** Very effective for CNNs; allows high learning rates.
- **Cons:**
   + Fails with small batch sizes *(noisy statistics)*.
   + Poor for RNNs/Transformers where sequence length varies or batch processing is complex.
   + Behavior differs between train *(batch stats)* and eval *(running stats)*, causing issues in fine-tuning or domain adaptation.

## Layer Normalization (LayerNorm)
Normalizes across the feature dimensions for each sample independently. Ignores the batch dimension.
- **Input Shape:** (N,T,D) typical for Transformers.
- **Statistics Calculation:**
  + For each sample n and time step t:  
   $\mu_{n,t} = \frac{1}{D} \sum_{d} x_{n,t,d}$
   $\sigma^2_{n,t} = \frac{1}{D} \sum_{d} (x_{n,t,d} - \mu_{n,t})^2$
- **Process:**
  + Compute mean/variance over the last dimensions D *(embedding dim)*.
  + Normalize.
  + Apply learnable $\gamma_d, \beta_d$ per feature dimension.
- **Pros:**
  + Batch-size invariant. Works with batch size 1.
  + Standard for Transformers (NLP).
- **Cons:**
  + Computationally slightly more expensive than BatchNorm for CNNs due to per-sample calculation.
  + Assumes features within a layer are correlated and should be normalized together.

## Instance Normalization (InstanceNorm)
Normalizes each channel of each sample independently. Removes contrast information from the content image. Originally designed for style transfer.
- **Input Shape:** (N,C,H,W)
- **Statistics Calculation:**
  + For each sample n and channel c:  
    $\mu_{n,c} = \frac{1}{H \cdot W} \sum_{h,w} x_{n,c,h,w}$
    $\sigma^2_{n,c} = \frac{1}{H \cdot W} \sum_{h,w} (x_{n,c,h,w} - \mu_{n,c})^2$
- **Process:**
  + Compute mean/variance over spatial dimensions $H, W$ only.
  + Normalize.
  + Apply learnable $\gamma_c, \beta_c$ per channel.
- **Pros:** Excellent for style transfer and generative models where global contrast/brightness should be removed.
- **Cons:** Loses global context information; not suitable for classification tasks where inter-sample statistics matter.

## Group Normalization (GroupNorm)
A compromise between BatchNorm and InstanceNorm. Divides channels into groups and normalizes within each group.
- **Input Shape:** (N, C, H, W)
- **Hyperparameter:** G *(number of groups, e.g., 32)*.
- **Statistics Calculation:**
  + Split C into G groups, each with C/G channels.
  + For each sample n and group g:  
    $\mu_{n,g} = \frac{1}{(C/G) \cdot H \cdot W} \sum_{c \in g, h, w} x_{n,c,h,w}$  
    $\sigma^2_{n,g} = \frac{1}{(C/G) \cdot H \cdot W} \sum_{c \in g, h, w} (x_{n,c,h,w} - \mu_{n,g})^2$
- **Process:**
  + Reshape/View tensor to separate groups.
  + Compute stats over (C/G, H, W).
  + Normalize.
  + Apply $\gamma, \beta$ per channel (or per group, depending on implementation, usually per channel).
- **Pros:**
  + Stable with small batch sizes.
  + Outperforms BatchNorm in object detection/segmentation *(where batch sizes are often small due to large images)*.
- **Cons:** Slightly more complex implementation than LayerNorm.

## RMSNorm (Root Mean Square Layer Normalization)
Simplifies LayerNorm by removing the mean subtraction. It assumes that re-centering *(subtracting mean)* is less important than re-scaling.
- **Formula:**
    $\text{RMS}(x) = \sqrt{\frac{1}{D} \sum_{i=1}^{D} x_i^2 + \epsilon}$  
    $\bar{x}_i = \frac{x_i}{\text{RMS}(x)}$  
    $y_i = \gamma_i \bar{x}_i$  

(Note: No β\betaβ shift parameter is typically used, though some variants include it.)
- **Process:**
  + Compute Root Mean Square over the feature dimension D.
  + Normalize by dividing by RMS.
  + Apply learnable scale γ\gammaγ.
- **Pros:**
  + ~7-64% faster than LayerNorm depending on hardware/kernel optimization because it skips mean calculation and subtraction.
  + Empirically equivalent to LayerNorm in Transformers *(LLaMA, PaLM use this)*.
- **Cons:** Theoretical justification for removing mean subtraction is weak, but empirical results hold.

## Pre-Norm vs. Post-Norm
This refers to the placement of the normalization layer relative to the residual connection and sub-layer *(Attention/MLP)* in Transformers.
- **Post-Norm (Original Transformer):**
   + ***Structure:*** Sublayer(x) $\rightarrow$ Add(x) $\rightarrow$ Norm()
   + ***Flow:*** Input $\rightarrow$ Attention $\rightarrow$ Add Residual $\rightarrow$ LayerNorm $\rightarrow$ Output
   + ***Characteristics:***
       * Stabilizes training in very deep networks initially.
       * Gradients have to pass through the residual addition before normalization. In very deep networks, this can lead to vanishing/exploding gradients because the norm constraint is applied after the accumulation.
       * Harder to train without warmup.
- **Pre-Norm (Standard for Modern LLMs):**
   + ***Structure:*** Norm(x) $\rightarrow$ Sublayer(Norm(x)) $\rightarrow$ Add(x)
   + ***Flow:*** Input $\rightarrow$ LayerNorm $\rightarrow$ Attention $\rightarrow$ Add Residual $\rightarrow$ Output
   + ***Characteristics:***
       * The gradient path through the residual branch is essentially identity *(unconstrained by weights/norms at the point of addition)*. This allows signals to propagate back through hundreds of layers easily.
       * Much more stable, allows removing learning rate warmup in some cases.
       * The output of the block is not normalized, which some argue preserves better representational capacity for the next layer.

## DeepNorm
A modification specifically designed to enable training of extremely deep Transformers *(1000+ layers)* without instability. It combines Pre-Norm structure with specific initialization and residual scaling.
- **Structure:** Similar to Pre-Norm, but modifies the residual connection weight.
- **Mechanism:**
  + Multiply the output of the sub-layer *(Attention/MLP)* by a constant $\alpha$ before adding to the residual.
    $x_{l+1} = x_l + \alpha \cdot \text{Sublayer}(\text{Norm}(x_l))$
  + Initialize the sub-layer weights with a specific variance scaling *(usually smaller than standard Xavier/He)* to counteract the depth.
  + NInitialize $\gamma$ in LayerNorm to a small value *(e.g., 0.1)* rather than 1.0.
- It controls the magnitude of the update at each layer, preventing the signal from exploding or vanishing over 1000+ layers. It effectively dampens the residual contribution early in training.
- Used for Extremely deep networks (>100 layers). For standard 12-96 layer models, standard Pre-Norm is usually sufficient and simpler.

## Sandwich Norm
Places normalization layers both before and after the sub-layer operation within the residual block.
- **Structure:**
    + $x^\prime = x + \text{Sublayer}(\text{Norm}_1(x))$   
    $x_{out} = \text{Norm}_2(x')$  
    + Or sometimes:   
    $x_{mid} = \text{Norm}_1(x)$  
    $x_{attn} = \text{Attention}(x_{mid})$  
    $x_{out} = \text{Norm}_2(x + x_{attn})$  
- **Process:**
  + Normalize input to sub-layer (Pre-Norm style).
  + Execute sub-layer.
  + Add residual.
  + Normalize the result again (Post-Norm style).
- **Pros:**
  + Pre-norm ensures good gradient flow into the block; Post-norm ensures the output passed to the next block is well-conditioned.
  + Can stabilize training in difficult regimes (e.g., high learning rates, unstable GANs/Diffusion).
- **Cons:**
  + Two normalization passes per block instead of one.
  + Can sometimes strip too much information, hurting performance if not tuned carefully.

## Filter Response Normalization
Designed specifically to replace BatchNorm in CNNs. It addresses the zero-mean problem where BatchNorm can sometimes shift activations too far from the origin, hurting ReLU-based networks. FRN normalizes per-channel but does not subtract the mean.
- **Input Shape:** (N,C,H,W)
- **Formula:**  
    $\tau_i = \frac{1}{H \cdot W} \sum_{h,w} x_{i,h,w}^2$   
    $\hat{x}_{i,h,w} = \frac{x_{i,h,w}}{\sqrt{\tau_i + \epsilon}}$   
    $y_{i,h,w} = \gamma_i \hat{x}_{i,h,w} + \beta_i$  
- **TLU (Thresholded Linear Unit):** FRN is almost always paired with TLU instead of ReLU. TLU prevents the dead neuron problem that can occur when FRN pushes values near zero.  
$\text{TLU}(x) = \max(x, \tau)$ where $\tau$ is a learnable threshold.
- **Pros:**
    + Works with batch size 1.
    + Outperforms BatchNorm in many CNN architectures *(ResNet, Inception)*.
    + No running averages needed for inference.
- **Cons:** Slightly more complex than RMSNorm due to the TLU requirement.

## Online Normalization
An algorithm that approximates BatchNorm statistics using an online moving average during the forward pass, but corrects for bias in the backward pass. It allows for accurate normalization even with tiny batch sizes without the noise of standard BatchNorm.
- **Process:**
  + **Forward Pass:** Maintain running estimates of mean ($\mu$) and variance ($\sigma^2$) updated at every step. Normalize current batch using these running stats.
  + **Backward Pass:** The gradient calculation is modified to account for the fact that the statistics were not computed from the current batch alone. It uses a control variate approach to reduce variance in the gradient estimates.
- **Pros:**
  + Extremely stable for small batches.
  + Removes the discrepancy between train and eval modes *(since it always uses running stats)*.
- **Cons:** High implementation complexity; requires careful tuning of the momentum for the running stats. Largely superseded by GroupNorm/RMSNorm in practice.

## Cross-Layer Normalization / Switchable Normalization
***Cross-Layer*** often refers to methods that share statistics across layers, but Switchable Normalization *(Luo et al., 2018)* is the more prominent technical implementation. Instead of choosing one norm *(Batch, Layer, or Instance)*, the network learns which normalization to use for each layer.
- **Formula:**
    $\hat{x} = \frac{x - \sum_{k} w_k \mu_k}{\sqrt{\sum_{k} w_k \sigma_k^2 + \epsilon}}$    
Where $k \in \{\text{Batch, Layer, Instance}\}$ and $w_k$ are learnable weights *(softmax-ed)* for each layer.  
- **Process:**
    + Compute $\mu$ and $\sigma$ for Batch, Layer, and Instance norms simultaneously.
    + Learn a weighted combination of these statistics.
    + Normalize using the combined stats.
- **Pros:** Robust across different tasks and batch sizes because the model adapts its normalization strategy.
- **Cons:** Significant computational overhead *(calculating three sets of stats per layer)*.

## PowerNorm
Addresses the issue that BatchNorm uses batch statistics which are noisy. PowerNorm uses running statistics for normalization but updates them in a way that maintains a consistent scale over time. It focuses on normalizing the power *(second moment)* of the activations.
- **Mechanism:**
  + Tracks the running average of squared activations ($E[x^2]$).
  + Normalizes by this running power.
  + Uses a recursive update rule for the statistics that is differentiable and stable.
- **Pros:**
  + Better than BatchNorm for small batches.
  + More stable than Online Norm.
- **Cons:** Complex update rules; largely experimental and less widely adopted than RMSNorm.

## Normalizer-Free Networks (NFNet)
Remove all normalization layers entirely. Instead, rely on adaptive gradient clipping and specialized initialization to stabilize training.
- **Adaptive Gradient Clipping:** Clips gradients based on the ratio of the gradient norm to the parameter norm ($\|\nabla\| / \|w\|$). This prevents large updates relative to the weight magnitude, stabilizing training without normalization.
- **SkipInit / Fixup-style Initialization:** Carefully scaling residual branches to ensure signal propagation.
- **Activation:** Uses GELU or similar smooth activations.
- **Pros:**
  + Removes the computational cost of calculating means/variances.
  + No hyperparameters for momentum or $\epsilon$ in norm layers.
  + Achieved SOTA on ImageNet at the time of publication.
- **Cons:** Requires careful implementation of AGC; sensitive to learning rate schedules.

## ReZero / SkipInit
These are initialization strategies that replace normalization by controlling the magnitude of the residual branch at the start of training.
- **ReZero:**
        Introduces a learnable scalar $\alpha$ initialized to 0 for each residual block.  
        $x_{l+1} = x_l + \alpha \cdot F(x_l)$   
        At initialization, the network is effectively an identity mapping. As training progresses, α\alphaα learns how much transformation to allow.
- **SkipInit:**
        Similar concept but uses a fixed scalar or a learned scalar initialized to a small value *(e.g., 0.1 or 1/$\sqrt{L}$)*.
    Often combined with standard weight initialization.
- **Pros:**
    + Allows training of extremely deep networks without normalization.
    + Simplifies the architecture *(no norm layers)*.
- **Cons:** Can be slower to converge initially compared to Pre-Norm because the network starts as an identity function and must learn to deviate.

## Fixup Initialization
A theoretical initialization scheme that allows training of deep ResNets without any normalization layers. It relies on the idea that if weights are scaled correctly, the variance of activations will remain stable through depth.
- Initialize all weights in residual branches to 0 for biases and $\mathcal{N}(0, \sigma^2)$ for weights, where σ\sigmaσ scales with $1/\sqrt{L}\;$ *(L = number of layers)*.
- Initialize the last layer of each residual block to 0.
- Use a specific learning rate warmup.
- **Pros:** Proves that normalization is not strictly necessary for optimization if initialization is handled correctly.
- **Cons:** Fragile; sensitive to architecture changes. Largely replaced by more robust methods like NFNet or DeepNorm.

## ActNorm (Activation Normalization)
Used in Normalizing Flows *(generative models)*. It is an invertible normalization layer that normalizes activations based on the first mini-batch of data.
- **Process:**
  + During the first forward pass, compute the mean and log-standard-deviation of the activations across the batch.
  + Initialize learnable parameters $\log(s)$ and bbb such that the output has zero mean and unit variance for that first batch.
  + For all subsequent steps, treat $\log(s)$ and bbb as standard learnable parameters.
- **Formula:**  
    $y = s \odot x + b$ 
- **Pros:**
  + Crucial for flow-based models where you need to compute the exact likelihood.
  + Stabilizes training of deep generative models.
- **Cons:**
  + The initial statistics depend on the first batch. If the first batch is unrepresentative, performance suffers.
  + Not suitable for general discriminative tasks *(classification/detection)*.

# Learning Rate Scheduling
The learning rate $\eta$ is the most sensitive hyperparameter in training. LR scheduling dynamically adjusts $\eta$ over time to balance exploration *(large steps early)* and convergence (small steps late). Without scheduling, training either explodes *(too high)* or plateaus prematurely *(too low)*.
- Step Decay
- MultiStep Decay
- Exponential Decay
- Polynomial Decay
- Cosine Annealing
- Warm Restarts (SGDR)
- Cyclical LR
- One Cycle Policy
- ReduceLROnPlateau
- Linear Warmup
- Exponential Warmup
- Constant Warmup
- Linear Scaling Rule
- Square Root Scaling
- Gradual Warmup
- Schedule-Free Optimizers
- Learned LR Scheduling
- LR as trainable parameter
- Warmup-Stable-Decay (WSD)
- Cosine with Restarts + Snapshot Ensembling

## Step Decay
- **Mechanism:** The learning rate is reduced by a fixed factor at regular, predetermined intervals *(epochs or iterations)*.
- **Formula:** $\eta_t = \eta_0 \times \gamma^{\lfloor t / s \rfloor}$
        + **$\eta_0$:** Initial learning rate
        + **$\gamma$:** Decay factor *(e.g., 0.1, 0.5)*
        + **s:** Step size *(number of epochs/iterations between drops)*
        + **t:** Current epoch/iteration
- **Characteristics:**
  + Creates a staircase pattern in the learning rate curve 
  + Simple to implement and predictable.
- **Drawback:** The abrupt drops can cause sudden spikes in loss or disrupt training momentum if the timing does not align with the model's convergence state 
- It requires manual tuning of the step size, which may not generalize across different architectures or datasets.

## MultiStep Decay
- **Mechanism:** A generalization of Step Decay where the learning rate is reduced by a factor at specific, non-uniform milestones rather than regular intervals.
- **Formula:** $\eta_t = \eta_0 \times \gamma^{\sum_{i} \mathbb{I}(t \geq m_i)}$
  + **$m_i$:** List of milestone epochs *(e.g., [30, 60, 90])*
  + **$\mathbb{I}$:** Indicator function *(1 if true, 0 otherwise)*
- **Characteristics:**
  + Allows for more flexible scheduling based on empirical observations of when the model plateaus.
  + Commonly used in standard computer vision benchmarks *(e.g., ResNet training often uses milestones at 30, 60, 90 epochs)*.
  + Like Step Decay, it suffers from abrupt transitions and requires prior knowledge of the training dynamics to set milestones effectively.

## Exponential Decay
- **Mechanism:** The learning rate decays continuously by multiplying it by a fixed factor after every epoch or iteration.
- **Formula:** $\eta_t = \eta_0 \times \gamma^t$
  + **$\gamma$:** Decay rate per step *(e.g., 0.95 or 0.99)*
- **Characteristics:**
  + Produces a smooth, monotonic decrease without abrupt jumps 
  + Ensures the learning rate approaches zero asymptotically but never reaches it exactly within finite steps.
- The decay factor $\gamma$ is sensitive; too high, and the rate stays large too long; too low, and it vanishes before convergence. It lacks the fine-tuning phase precision of cosine schedules because the rate of change is constant in log-space.

## Polynomial Decay
- **Mechanism:** The learning rate decays according to a polynomial function of the current step relative to the total steps.
- **Formula:** $\eta_t = \eta_{end} + (\eta_0 - \eta_{end}) \times (1 - \frac{t}{T})^p$
  + **$\eta_{end}$:** Final learning rate *(often 0)*
  + **T:** Total number of steps/epochs
  + **p:** Power coefficient *(e.g., p=1 is linear decay, p=2 is quadratic)*
- **Characteristics:**
  + Flexible shape depending on power p.
  + Linear decay ($p=1$) decreases rapidly at first and slows down, which may be suboptimal as early stages often benefit from sustained higher rates for exploration 
  + Higher powers create a long tail where the learning rate stays relatively high for longer before dropping sharply near the end.

## Cosine Annealing
- **Mechanism:** The learning rate follows a cosine curve, decreasing smoothly from an initial maximum to a minimum over a fixed period.
- **Formula:** $\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\frac{t \pi}{T}))$
  + **T:** Total steps in the cycle
  + **t:** Current step ($0 \le t \le T$)
- **Characteristics:**
  + Avoids the shocks of step decay. The derivative is zero at the start and end, allowing for gentle transitions 
  + Keeps the learning rate higher for longer in the early phase compared to linear decay, aiding exploration, then drops rapidly in the middle, and slows down again at the end for fine-tuning 
- Widely considered a strong default for modern deep learning tasks due to its robustness and lack of hyperparameter sensitivity regarding when to drop the rate.

## Warm Restarts (Stochastic Gradient Descent with Warm Restarts)
- **Mechanism:** Combines Cosine Annealing with periodic restarts. After each cosine cycle completes (reaching $\eta_{min}$), the learning rate is reset to $\eta_{max}$ and a new cosine cycle begins.
- **Formula:** Same as Cosine Annealing, but t is reset modulo the cycle length $T_i$. The cycle length $T_i$ can increase by a factor $T_{mult}$ after each restart.
- **Characteristics:**
  + The periodic increase in learning rate helps the optimizer escape shallow local minima or saddle points, potentially finding wider, more generalizable minima 
  + Provides good checkpoints at the end of each cycle.
  + In PyTorch, `CosineAnnealingWarmRestarts` implements this. Note that standard `CosineAnnealingLR` does not include restarts unless manually looped

## Cyclical Learning Rates
- **Mechanism:** The learning rate oscillates cyclically between a lower bound ($\eta_{min}$) and an upper bound ($\eta_{max}$) without necessarily decaying the bounds over time *(though they can)*.
- **Common Shapes:** Triangular, Triangular2 (amplitude halves each cycle), ExpRange (exponential decay of bounds).
- **Triangular Formula:** Linearly increases from $\eta_{min}$ to $\eta_{max}$ over half the cycle, then linearly decreases back.
- **Characteristics:**
  + Proposed by Leslie Smith to address the difficulty of tuning the initial learning rate 
  + Periodic increases help traverse saddle points and explore the loss surface more effectively than monotonically decreasing schedules 
- Requires determining appropriate bounds via a Learning Rate Range Test *(train with increasing LR and observe loss)*.

## One Cycle Policy
- **Mechanism:** A specific cyclical schedule that spans the entire training duration *(one single cycle)*. It typically consists of:
    + **Warmup:** Linear increase from a low LR to a high peak LR ($\eta_{max}$).
    + **Annealing:** Decrease from $\eta_{max}$ to a very low LR ($\eta_{min}$), often using cosine annealing.
- **Characteristics:**
  + Can achieve faster training times and higher accuracy by allowing larger learning rates for part of the training 
  + Often paired with inverse momentum scheduling *(high momentum when LR is low, low momentum when LR is high)* to stabilize training.
  + Simpler than multi-cycle CLR as it has no repeating cycles; the entire training run is one arc.

## ReduceLROnPlateau
- **Mechanism:** An adaptive scheduler that monitors a metric *(usually validation loss)* and reduces the learning rate by a factor when the metric stops improving.
- **Parameters:**
  + **Patience:** Number of epochs with no improvement before reducing LR.
  + **Factor:** Multiplicative factor for reduction *(e.g., 0.1)*.
  + **Threshold:** Minimum change in metric to qualify as improvement.
- **Characteristics:**
  + Does not follow a predetermined time-based schedule but responds to actual training dynamics 
  + Useful when the optimal schedule is unknown.
  + Can be slow to react if patience is high. It may reduce the LR too late if the model has already stagnated. It requires a validation set, making it less suitable for purely unsupervised or online learning scenarios without modification.

## Linear Warmup
- **Mechanism:** The learning rate starts at 0 *(or a very small value)* and increases linearly to the target initial learning rate ($\eta_0$) over a specified number of steps ($T_{warmup}$).
- **Formula:** $\eta_t = \eta_0 \times \frac{t}{T_{warmup}}$ for $t < T_{warmup}$
- **Characteristics:**
    + Crucial for training large models *(e.g., Transformers)* or using large batch sizes. Prevents divergence caused by large gradients in the early stages when parameters are randomly initialized 
    + Standard practice in modern NLP and Vision Transformers (ViT). Usually followed by a decay schedule *(e.g., Cosine or Step)*.

## Exponential Warmup
- **Mechanism:** The learning rate increases exponentially from a small value to the target η0\eta_0η0​ during the warmup phase.
- **Formula:** $\eta_t = \eta_{start} \times (\frac{\eta_0}{\eta_{start}})^{\frac{t}{T_{warmup}}}$
- **Characteristics:**
  + Less common than linear warmup.
  + Provides a slower initial increase, which might be beneficial if the initial gradients are extremely volatile, but linear warmup is generally sufficient and simpler.

## Constant Warmup
- **Mechanism:** The learning rate is held at a small constant value for the first $T_{warmup}$ steps, then abruptly jumps to the target $\eta_0$.
- **Characteristics:**
  + **Not Recommended:** The abrupt jump from a low constant rate to the full learning rate can cause instability and gradient explosions, defeating the purpose of warmup 
- Gradual warmup *(linear or exponential)* is preferred to smooth the transition into the main training phase.

## Linear Scaling Rule
- **Mechanism:** A heuristic for adjusting the learning rate when increasing the mini-batch size. It posits that the learning rate should be scaled linearly with the batch size to maintain the same signal-to-noise ratio in the gradient estimates.
- **Formula:** $\eta_{new} = \eta_{base} \times \frac{B_{new}}{B_{base}}$
  + $\eta_{base}$: Original learning rate
  + $B_{base}$: Original batch size
  + $B_{new}$: New, larger batch size
- **Characteristics:**
  + Assumes that the gradient variance scales inversely with batch size. By increasing the LR linearly, the step size relative to the noise level remains constant.
  + Breaks down at extremely large batch sizes where the loss landscape geometry changes, often requiring additional techniques like Layer Normalization or specialized optimizers *(e.g., LAMB)* to remain stable .

## Square Root Scaling
- **Mechanism:** An alternative scaling rule suggesting that the learning rate should scale with the square root of the batch size.
- **Formula:** $\eta_{new} = \eta_{base} \times \sqrt{\frac{B_{new}}{B_{base}}}$
- **Characteristics:** Derived from the central limit theorem and stochastic approximation theory, which suggest that the standard deviation of the gradient noise scales with $1/\sqrt{B}$. To keep the signal-to-noise ratio constant, the step size should scale with $\sqrt{B}$.
- **Usage:** Often observed in Bayesian optimization contexts or smaller-scale experiments. It is more conservative than linear scaling, reducing the risk of divergence when scaling up batch sizes moderately.

## Gradual Warmup
- **Mechanism:** A general term for strategies that slowly increase the learning rate from a near-zero value to the target initial learning rate over a predefined period. This includes Linear and Exponential warmups but emphasizes the purpose: stabilizing the early phase of training.
- Typically applied for the first 5–10% of total training steps.
- In Transformer models, it is often combined with an inverse square root decay after the warmup phase *(e.g., *\eta_t = \eta_0 \cdot \min(\frac{t}{T_{warmup}}, \frac{1}{\sqrt{t}})$))*.
- **Characteristics:**
    + Prevents gradient explosion in the first few steps when weights are randomly initialized and gradients can be erratic.
    + Essential for distributed training where large effective batch sizes are used.

## Schedule-Free Optimizers
- **Mechanism:** A recent class of optimizers *(e.g., Schedule-Free AdamW, Schedule-Free SGD)* that eliminate the need for explicit learning rate schedules. Instead, they use a combination of momentum and averaging to achieve convergence without manual LR tuning.
  + The optimizer maintains two sets of parameters: the current iterate ($x_t$) and an averaged sequence ($z_t$). The model weights used for forward passes are an interpolation of these two.
  +  Uses a fixed learning rate throughout training.
- **Characteristics:**
    + Removes hyperparameters like decay rates, milestones, or cycle lengths.
    + Empirically matches or exceeds tuned schedule-based methods on various benchmarks *(NLP, Vision)* by implicitly adapting the effective step size through the averaging mechanism .
    + Requires careful tuning of the interpolation coefficient ($\beta$) and weight decay, but removes the complexity of LR scheduling entirely.

## Learned LR Scheduling
- **Mechanism:** Uses a meta-learning approach where a neural network *(controller)* predicts the optimal learning rate at each step based on the training history *(losses, gradients, previous LRs)*.
- **Reinforcement Learning:** The controller is trained via policy gradients to maximize validation accuracy.
- **Recurrent Networks:** An RNN/LSTM takes the training trajectory as input and outputs the next LR.
- **Characteristics:**
  + Can adapt to complex, non-stationary loss landscapes where fixed formulas fail.
  + Computationally expensive due to the overhead of running the controller and the meta-training process.
  + Learned schedulers often struggle to generalize across different architectures or datasets without re-training the controller.

## LR as Trainable Parameter
- **Mechanism:** Treats the learning rate *(or its log)* as a trainable parameter within the optimization graph, updating it via gradient descent alongside the model weights.
- **Implementation:**
  + Define $\log(\eta)$ as a parameter.
  + Compute gradients of the loss with respect to $\eta$ *(hypergradients)*.
  + Update $\eta$ using a separate, small learning rate.
- Characteristics:
  + Allows the LR to adjust dynamically to local curvature.
  + Directly differentiating through the optimization steps is noisy. Techniques like ***Hypergradient Descent*** or using a validation loss for updates are required to stabilize training .
  + Increases the dimensionality of the optimization problem and can lead to oscillatory behavior if not carefully regularized.

## Warmup-Stable-Decay (WSD)
- **Mechanism:** A three-phase schedule specifically designed for large language model *(LLM)* pre-training, especially when training data or compute budget may be extended unexpectedly.
- **Phases:**
  + **Warmup:** Linear increase to $\eta_{max}$.
  + **Stable:** Maintain $\eta_{max}$ for the majority of training *(e.g., 70–80%)*.
  + **Decay:** Rapid decay *(often cosine or linear)* to zero in the final phase.
- **Characteristics:**
  + Unlike Cosine Annealing, which requires knowing the total steps T in advance, WSD allows training to be stopped or extended during the ***Stable*** phase without significant performance degradation.
  + If more data becomes available, training can continue at the stable high LR, then a new decay phase can be appended. This is crucial for LLMs where training duration is often uncertain .

## Cosine with Restarts + Snapshot Ensembling
- **Mechanism:** Combines Stochastic Gradient Descent with Warm Restarts (SGDR) with the practice of saving model checkpoints *(snapshots)* at the end of each cycle.
- **Process:**
  + Train with Cosine Annealing with Warm Restarts.
  + At the end of each cycle *(when LR reaches $\eta_{min}$)*, save the model weights as a snapshot.
  + Reset LR to $\eta_{max}$ and start the next cycle.
  + At inference, average the predictions of all saved snapshots.
- **Characteristics:**
  + Each snapshot converges to a different local minimum because the high LR restarts force the optimizer to explore new regions of the loss landscape.
  + Averaging diverse models reduces variance and improves generalization without the cost of training multiple independent models from scratch .
  + Achieves ensemble-like performance for the cost of a single long training run.